# 08.最终模型结论整理

汇总 LGBM/XGB 结果并生成最终推荐结论。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

In [ ]:
# 示例表：实际项目中可从 05/07 输出文件读取后自动拼接
final_model_summary = pd.DataFrame([
    {"算法": "LGBM", "建模标签": "y1", "模型版本": "不含评分", "OOT_KS": 0.265075, "OOT_AUC": 0.670954, "衰减是否达标": "是", "baseline是否达标": "是"},
    {"算法": "LGBM", "建模标签": "y1", "模型版本": "含评分_4A", "OOT_KS": 0.293094, "OOT_AUC": 0.700795, "衰减是否达标": "否", "baseline是否达标": "是"},
    {"算法": "LGBM", "建模标签": "y1", "模型版本": "含评分_4B", "OOT_KS": 0.294041, "OOT_AUC": 0.701583, "衰减是否达标": "否", "baseline是否达标": "是"},
    {"算法": "LGBM", "建模标签": "y2", "模型版本": "不含评分", "OOT_KS": 0.168198, "OOT_AUC": 0.591065, "衰减是否达标": "是", "baseline是否达标": "否"},
    {"算法": "LGBM", "建模标签": "y2", "模型版本": "含评分_4A", "OOT_KS": 0.226409, "OOT_AUC": 0.617898, "衰减是否达标": "是", "baseline是否达标": "是"},
    {"算法": "LGBM", "建模标签": "y2", "模型版本": "含评分_4B", "OOT_KS": 0.226409, "OOT_AUC": 0.617920, "衰减是否达标": "是", "baseline是否达标": "是"},
    {"算法": "XGB", "建模标签": "y1", "模型版本": "不含评分", "OOT_KS": 0.243054, "OOT_AUC": 0.676987, "衰减是否达标": "否", "baseline是否达标": "是"},
    {"算法": "XGB", "建模标签": "y1", "模型版本": "含评分_4A", "OOT_KS": 0.279228, "OOT_AUC": 0.704881, "衰减是否达标": "否", "baseline是否达标": "是"},
    {"算法": "XGB", "建模标签": "y1", "模型版本": "含评分_4B", "OOT_KS": 0.279228, "OOT_AUC": 0.704881, "衰减是否达标": "否", "baseline是否达标": "是"},
    {"算法": "XGB", "建模标签": "y2", "模型版本": "不含评分", "OOT_KS": 0.165911, "OOT_AUC": 0.588875, "衰减是否达标": "是", "baseline是否达标": "否"},
    {"算法": "XGB", "建模标签": "y2", "模型版本": "含评分_4A", "OOT_KS": 0.279336, "OOT_AUC": 0.695792, "衰减是否达标": "否", "baseline是否达标": "是"},
    {"算法": "XGB", "建模标签": "y2", "模型版本": "含评分_4B", "OOT_KS": 0.279336, "OOT_AUC": 0.695792, "衰减是否达标": "否", "baseline是否达标": "是"},
])

final_rank = final_model_summary.sort_values("OOT_KS", ascending=False).reset_index(drop=True)
final_rank["最终排名"] = final_rank.index + 1

def build_reason(row):
    reason = []
    if row["OOT_KS"] >= 0.25:
        reason.append("OOT_KS较高")
    if row["baseline是否达标"] == "是":
        reason.append("达到baseline要求")
    if str(row["模型版本"]).startswith("含评分"):
        reason.append("评分变量有效提升模型")
    if row["衰减是否达标"] == "是":
        reason.append("模型稳定性较好")
    return "；".join(reason)

final_rank["推荐理由"] = final_rank.apply(build_reason, axis=1)

display(final_rank)

In [ ]:
best_model = final_rank.iloc[0]

final_conclusion = pd.DataFrame([{
    "最终推荐算法": best_model["算法"],
    "最终推荐标签": best_model["建模标签"],
    "最终推荐模型版本": best_model["模型版本"],
    "最终OOT_KS": best_model["OOT_KS"],
    "最终OOT_AUC": best_model["OOT_AUC"],
    "总体结论": "含评分模型整体优于不含评分模型；LGBM在y1含评分场景下表现最好；XGB在y2含评分场景下有明显提升，可作为备选或后续优化方向。"
}])

display(final_conclusion)

report_path = OUTPUT_DIR / f"8.最终模型结论整理_{customer_type}.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    final_model_summary.to_excel(writer, sheet_name="1_全部模型结果", index=False)
    final_rank.to_excel(writer, sheet_name="2_最终模型排序", index=False)
    final_conclusion.to_excel(writer, sheet_name="3_最终推荐结论", index=False)

print("saved:", report_path)